# Telegram NLP Pipeline (Colab)

Этот ноутбук обучает/настраивает модели и делает 4 задачи:
1) Topic classification (категория поста)
2) NER (PER/ORG/LOC, фокус на PER)
3) Sentiment
4) Theme clustering + same-theme scoring

Режимы:
- A) без разметки: zero-shot + кластеризация
- B) с разметкой: fine-tune topic/sentiment

Запускать сверху вниз в Google Colab.

## A) Colab Setup

Установка зависимостей, проверка GPU, фиксация random seeds, настройка путей.

In [ ]:
!pip -q install pandas numpy scikit-learn transformers datasets evaluate sentence-transformers umap-learn hdbscan natasha razdel pymorphy2 networkx pyvis matplotlib joblib tqdm

In [ ]:
import os
import re
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
PROJECT_NAME = "tg_news_project"
MOUNT_DRIVE = False  # поставьте True, если хотите сохранять в Google Drive

BASE_DIR = Path("/content")
if MOUNT_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        if Path("/content/drive/MyDrive").exists():
            BASE_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as exc:
        print("Drive mount failed, using /content:", exc)

OUT_DIR = BASE_DIR / "outputs"
ART_DIR = BASE_DIR / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

# По умолчанию используем обучающий датасет Lenta.ru
DATASET_MODE = "lenta"  # "lenta" или "telegram"
if DATASET_MODE == "lenta":
    DATA_PATH = Path("/content/lenta-ru-news.csv.gz")
else:
    DATA_PATH = Path("/content/telegram_posts.parquet")  # или /content/telegram_posts.csv

print("BASE_DIR:", BASE_DIR)
print("OUT_DIR:", OUT_DIR)
print("ART_DIR:", ART_DIR)
print("DATA_PATH:", DATA_PATH)
print("DATASET_MODE:", DATASET_MODE)

## B) Data Loading + Cleaning + EDA

Загрузка CSV/Parquet, очистка, сэмплирование (опционально) и EDA-графики.

### Загрузка обучающего датасета Lenta.ru (по умолчанию)

Этот блок обязателен, если `DATASET_MODE = "lenta"`.

In [ ]:
!wget -q -O lenta-ru-news.csv.gz \
  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

import pandas as pd

df = pd.read_csv("lenta-ru-news.csv.gz")
df.head(), df.columns, df.shape

In [ ]:
TOPIC_CANDIDATE_COLS = ["topic", "label", "category", "topic_label"]
SENTIMENT_CANDIDATE_COLS = ["sentiment", "sentiment_label"]


def detect_label_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def load_data(
    data_path: Path,
    usecols: list[str] | None = None,
    max_rows: int | None = None,
) -> tuple[pd.DataFrame, Path]:
    if data_path.exists():
        if data_path.suffix == ".parquet":
            df = pd.read_parquet(data_path, columns=usecols)
            if max_rows:
                df = df.head(max_rows)
            return df, data_path
        if data_path.suffix == ".csv":
            return pd.read_csv(data_path, usecols=usecols, nrows=max_rows), data_path
        if data_path.suffix == ".gz" or str(data_path).endswith(".csv.gz"):
            return (
                pd.read_csv(data_path, compression="gzip", usecols=usecols, nrows=max_rows),
                data_path,
            )

    alt_parquet = data_path.with_suffix(".parquet")
    alt_csv = data_path.with_suffix(".csv")
    alt_gz = data_path.with_suffix(".csv.gz")
    if alt_parquet.exists():
        df = pd.read_parquet(alt_parquet, columns=usecols)
        if max_rows:
            df = df.head(max_rows)
        return df, alt_parquet
    if alt_csv.exists():
        return pd.read_csv(alt_csv, usecols=usecols, nrows=max_rows), alt_csv
    if alt_gz.exists():
        return (
            pd.read_csv(alt_gz, compression="gzip", usecols=usecols, nrows=max_rows),
            alt_gz,
        )

    raise FileNotFoundError(
        f"Файл данных не найден: {data_path} (или {alt_parquet}, {alt_csv})"
    )


def normalize_whitespace(text: str) -> str:
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "post_id" not in df.columns:
        df["post_id"] = np.arange(len(df))

    if "channel" not in df.columns:
        df["channel"] = "unknown"

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        df["date"] = pd.NaT

    if "text" not in df.columns:
        raise ValueError("Ожидается колонка 'text'")

    df["text"] = df["text"].astype(str)
    if "title" in df.columns:
        df["title"] = df["title"].fillna("").astype(str)
        df["text_full"] = (
            df["title"].str.strip() + ". " + df["text"].str.strip()
        ).str.strip(". ")
    else:
        df["text_full"] = df["text"]

    df["text_clean"] = df["text_full"].apply(normalize_whitespace)
    df = df[df["text_clean"].str.len() >= 20].copy()

    topic_col = detect_label_col(df, TOPIC_CANDIDATE_COLS)
    if topic_col:
        df["topic_label"] = df[topic_col].astype(str).str.strip()
        df.loc[
            df["topic_label"].isin(["", "nan", "None"]) | df["topic_label"].isna(),
            "topic_label",
        ] = np.nan
    else:
        df["topic_label"] = np.nan

    sentiment_col = detect_label_col(df, SENTIMENT_CANDIDATE_COLS)
    if sentiment_col:
        df["sentiment_label"] = df[sentiment_col].astype(str).str.strip()
        df.loc[
            df["sentiment_label"].isin(["", "nan", "None"]) | df["sentiment_label"].isna(),
            "sentiment_label",
        ] = np.nan
    else:
        df["sentiment_label"] = np.nan

    dedup_cols = ["post_id"] if "post_id" in df.columns else ["text_clean"]
    df = df.drop_duplicates(subset=dedup_cols)

    return df


def run_eda(df: pd.DataFrame) -> None:
    print("Rows:", len(df))
    print("Unique channels:", df["channel"].nunique())

    if df["date"].notna().any():
        print("Date range:", df["date"].min(), "→", df["date"].max())

    df["text_len"] = df["text_clean"].str.len()

    plt.figure(figsize=(6, 4))
    plt.hist(df["text_len"], bins=50, color="#4472c4")
    plt.title("Text length distribution")
    plt.xlabel("Characters")
    plt.ylabel("Count")
    plt.show()

    top_channels = df["channel"].value_counts().head(20)
    plt.figure(figsize=(8, 4))
    top_channels.plot(kind="bar", color="#5b9bd5")
    plt.title("Top channels by volume")
    plt.ylabel("Posts")
    plt.show()

    if df["date"].notna().any():
        ts = df.set_index("date").resample("D").size()
        plt.figure(figsize=(8, 4))
        ts.plot(color="#70ad47")
        plt.title("Posts over time (daily)")
        plt.ylabel("Posts")
        plt.show()

    if df["topic_label"].notna().any():
        topic_counts = df["topic_label"].value_counts()
        display(topic_counts.to_frame("count"))
        plt.figure(figsize=(8, 4))
        topic_counts.plot(kind="bar", color="#ed7d31")
        plt.title("Topic label distribution")
        plt.ylabel("Posts")
        plt.show()

    if df["sentiment_label"].notna().any():
        sent_counts = df["sentiment_label"].value_counts()
        display(sent_counts.to_frame("count"))
        plt.figure(figsize=(6, 4))
        sent_counts.plot(kind="bar", color="#a5a5a5")
        plt.title("Sentiment label distribution")
        plt.ylabel("Posts")
        plt.show()

In [ ]:
USE_SAMPLE = DATASET_MODE == "lenta"
SAMPLE_SIZE = 20000
MAX_ROWS = None  # например 200000, если упираетесь в RAM

load_usecols = ["text", "topic", "title"] if DATASET_MODE == "lenta" else None
raw_df, used_path = load_data(DATA_PATH, usecols=load_usecols, max_rows=MAX_ROWS)
print("Loaded:", used_path, "shape=", raw_df.shape)

if USE_SAMPLE and len(raw_df) > SAMPLE_SIZE:
    raw_df = raw_df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print("Sampled raw:", raw_df.shape)

if DATASET_MODE == "lenta" and "channel" not in raw_df.columns:
    raw_df["channel"] = "lenta.ru"

df = clean_data(raw_df)
print("After clean:", df.shape)

if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print("Sampled clean:", df.shape)

(df).to_parquet(OUT_DIR / "telegram_clean.parquet", index=False)
run_eda(df)

## C) Category (topic)

Если есть разметка — fine-tune RuBERT. Если нет — zero-shot.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)
from datasets import Dataset
import evaluate
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

TOPIC_ZERO_SHOT_MODEL = "joeddav/xlm-roberta-large-xnli"
TOPIC_FINETUNE_MODEL = "DeepPavlov/rubert-base-cased"

TOPIC_LABELS = [
    "политика",
    "экономика",
    "общество",
    "технологии",
    "спорт",
    "культура",
    "происшествия",
    "здоровье",
    "безопасность/война",
    "наука",
    "бизнес",
]


def softmax_np(logits: np.ndarray, axis: int = 1) -> np.ndarray:
    logits = logits - logits.max(axis=axis, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=axis, keepdims=True)


def train_text_classifier(
    df: pd.DataFrame,
    text_col: str,
    label_col: str,
    model_name: str,
    art_dir: Path,
    num_epochs: int = 2,
) -> tuple[Trainer, AutoTokenizer, dict, dict, pd.DataFrame]:
    art_dir = Path(art_dir)
    art_dir.mkdir(parents=True, exist_ok=True)

    train_df = df[[text_col, label_col]].dropna().copy()
    train_df["row_id"] = np.arange(len(train_df))

    labels = sorted(train_df[label_col].unique())
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    train_df["label_id"] = train_df[label_col].map(label2id)

    from sklearn.model_selection import train_test_split
    from datasets import DatasetDict

    label_counts = train_df["label_id"].value_counts()
    can_stratify = label_counts.min() >= 2

    train_split, test_split = train_test_split(
        train_df,
        test_size=0.1,
        random_state=42,
        stratify=train_df["label_id"] if can_stratify else None,
    )

    dataset = DatasetDict(
        {
            "train": Dataset.from_pandas(train_split, preserve_index=False),
            "test": Dataset.from_pandas(test_split, preserve_index=False),
        }
    )

    raw_test_df = dataset["test"].to_pandas()

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=256,
        )

    ds = dataset.map(tokenize, batched=True)
    ds = ds.rename_column("label_id", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    acc = evaluate.load("accuracy")
    f1 = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {
            "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
            "f1": f1.compute(predictions=preds, references=labels, average="macro")[
                "f1"
            ],
        }

    def make_training_args(**kwargs):
        import inspect

        sig = inspect.signature(TrainingArguments.__init__)
        params = sig.parameters

        if (
            "evaluation_strategy" in kwargs
            and "evaluation_strategy" not in params
            and "eval_strategy" in params
        ):
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")

        if "save_strategy" in kwargs and "save_strategy" not in params:
            kwargs.pop("save_strategy", None)

        filtered = {k: v for k, v in kwargs.items() if k in params}

        if (
            "evaluation_strategy" not in params
            and "eval_strategy" not in params
            and "do_eval" in params
        ):
            filtered["do_eval"] = True

        return TrainingArguments(**filtered)

    args = make_training_args(
        output_dir=str(art_dir / "checkpoints"),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=42,
    )

    def make_trainer(**kwargs):
        import inspect

        sig = inspect.signature(Trainer.__init__)
        params = sig.parameters
        filtered = {k: v for k, v in kwargs.items() if k in params}
        return Trainer(**filtered)

    trainer = make_trainer(
        model=model,
        args=args,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    trainer.save_model(str(art_dir))
    tokenizer.save_pretrained(str(art_dir))

    with open(art_dir / "label2id.json", "w", encoding="utf-8") as f:
        json.dump(label2id, f, ensure_ascii=False, indent=2)
    with open(art_dir / "id2label.json", "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in id2label.items()}, f, ensure_ascii=False, indent=2)

    return trainer, tokenizer, label2id, id2label, raw_test_df


def zero_shot_topic(
    df: pd.DataFrame,
    text_col: str,
    candidate_labels: list[str],
    model_name: str,
    batch_size: int = 16,
) -> tuple[pd.DataFrame, str]:
    device = 0 if torch.cuda.is_available() else -1
    clf = pipeline("zero-shot-classification", model=model_name, device=device)

    labels = []
    scores = []
    for i in tqdm(range(0, len(df), batch_size)):
        batch_texts = df[text_col].iloc[i : i + batch_size].tolist()
        outputs = clf(batch_texts, candidate_labels, multi_label=False)
        for out in outputs:
            labels.append(out["labels"][0])
            scores.append(float(out["scores"][0]))

    pred_df = pd.DataFrame(
        {
            "post_id": df["post_id"].values,
            "topic_label": labels,
            "topic_score": scores,
        }
    )
    return pred_df, model_name


def train_or_zero_shot_topic(
    df: pd.DataFrame,
    text_col: str = "text_clean",
    label_col: str = "topic_label",
) -> tuple[pd.DataFrame, str]:
    if df[label_col].notna().any():
        trainer, tokenizer, label2id, id2label, raw_test_df = train_text_classifier(
            df,
            text_col=text_col,
            label_col=label_col,
            model_name=TOPIC_FINETUNE_MODEL,
            art_dir=ART_DIR / "topic_clf",
            num_epochs=2,
        )

        preds = trainer.predict(trainer.eval_dataset)
        y_true = preds.label_ids
        y_pred = np.argmax(preds.predictions, axis=1)

        label_ids = list(range(len(id2label)))
        cm = confusion_matrix(y_true, y_pred, labels=label_ids)
        disp = ConfusionMatrixDisplay(
            cm, display_labels=[id2label[i] for i in label_ids]
        )
        fig, ax = plt.subplots(figsize=(8, 6))
        disp.plot(ax=ax, xticks_rotation=45)
        plt.title("Topic confusion matrix")
        plt.show()

        error_idx = np.where(y_true != y_pred)[0][:10]
        if len(error_idx) > 0:
            errors_df = raw_test_df.iloc[error_idx].copy()
            errors_df["pred_label"] = [id2label[i] for i in y_pred[error_idx]]
            errors_df["true_label"] = [id2label[i] for i in y_true[error_idx]]
            display(errors_df[[text_col, "true_label", "pred_label"]])

        full_ds = Dataset.from_pandas(df[[text_col]], preserve_index=False)
        tokenizer = AutoTokenizer.from_pretrained(str(ART_DIR / "topic_clf"))

        def tokenize(batch):
            return tokenizer(
                batch[text_col],
                truncation=True,
                padding="max_length",
                max_length=256,
            )

        full_ds = full_ds.map(tokenize, batched=True)
        full_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
        full_preds = trainer.predict(full_ds)
        probs = softmax_np(full_preds.predictions)
        pred_ids = probs.argmax(axis=1)

        pred_df = pd.DataFrame(
            {
                "post_id": df["post_id"].values,
                "topic_label": [id2label[i] for i in pred_ids],
                "topic_score": probs.max(axis=1),
            }
        )
        return pred_df, TOPIC_FINETUNE_MODEL

    pred_df, model_name = zero_shot_topic(
        df, text_col=text_col, candidate_labels=TOPIC_LABELS, model_name=TOPIC_ZERO_SHOT_MODEL
    )
    return pred_df, model_name

In [ ]:
topic_pred_df, topic_model_name = train_or_zero_shot_topic(df)

(topic_pred_df).to_parquet(OUT_DIR / "topic_predictions.parquet", index=False)

# сохраняем gold-метки отдельно, а дальше работаем с предсказаниями
if "topic_label_gold" not in df.columns:
    df["topic_label_gold"] = df["topic_label"]

pred_cols = topic_pred_df.rename(
    columns={"topic_label": "topic_label_pred", "topic_score": "topic_score"}
)

df = df.merge(pred_cols, on="post_id", how="left")
df["topic_label"] = df["topic_label_pred"].fillna(df["topic_label"])

display(df["topic_label"].value_counts().to_frame("count"))
plt.figure(figsize=(8, 4))
(df["topic_label"].value_counts()).plot(kind="bar", color="#ffc000")
plt.title("Predicted topic distribution")
plt.ylabel("Posts")
plt.show()

print("Topic model:", topic_model_name)

## D) Sentiment

Fine-tune при наличии разметки, иначе — pretrained модель.

In [ ]:
SENTIMENT_PRETRAINED_MODEL = "blanchefort/rubert-base-cased-sentiment"
SENTIMENT_FINETUNE_MODEL = "DeepPavlov/rubert-base-cased"

SENTIMENT_CONFIG = {
    "model_name": SENTIMENT_PRETRAINED_MODEL,
    "max_length": 384,
    "chunk_overlap": 64,
    "use_chunking": True,
    "neutral_threshold": 0.55,
    "neutral_label": "neutral",
    "label_map": {
        "LABEL_0": "negative",
        "LABEL_1": "neutral",
        "LABEL_2": "positive",
        "NEGATIVE": "negative",
        "NEUTRAL": "neutral",
        "POSITIVE": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive",
    },
}


def normalize_sentiment_label(label: str, label_map: dict) -> str:
    if label in label_map:
        return label_map[label]
    lower = label.lower()
    if lower in label_map:
        return label_map[lower]
    return lower


def _aggregate_logits_for_text(
    text: str,
    tokenizer,
    model,
    device,
    max_length: int,
    chunk_overlap: int,
) -> np.ndarray:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if not token_ids:
        return np.zeros(model.config.num_labels, dtype=float)

    specials = tokenizer.num_special_tokens_to_add(pair=False)
    chunk_size = max(1, max_length - specials)
    overlap = min(chunk_overlap, max(0, chunk_size - 1))
    step = max(1, chunk_size - overlap)

    logits_list = []
    for start in range(0, len(token_ids), step):
        chunk = token_ids[start : start + chunk_size]
        input_ids = tokenizer.build_inputs_with_special_tokens(chunk)
        attention_mask = [1] * len(input_ids)
        inputs = {
            "input_ids": torch.tensor([input_ids], device=device),
            "attention_mask": torch.tensor([attention_mask], device=device),
        }
        with torch.inference_mode():
            outputs = model(**inputs)
        logits_list.append(outputs.logits.detach().cpu().numpy()[0])
        if start + chunk_size >= len(token_ids):
            break

    return np.mean(logits_list, axis=0)


def pretrained_sentiment(
    df: pd.DataFrame,
    text_col: str,
    model_name: str,
    batch_size: int = 32,
    config: dict = SENTIMENT_CONFIG,
) -> tuple[pd.DataFrame, str]:
    labels = []
    scores = []
    label_map = config.get("label_map", {})
    neutral_threshold = config.get("neutral_threshold", None)
    neutral_label = config.get("neutral_label", "neutral")

    if config.get("use_chunking", True):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
        model.eval()

        max_length = int(config.get("max_length", 256))
        chunk_overlap = int(config.get("chunk_overlap", 64))

        for text in tqdm(df[text_col].tolist()):
            logits = _aggregate_logits_for_text(
                text, tokenizer, model, device, max_length, chunk_overlap
            )
            probs = softmax_np(np.array([logits]))[0]
            pred_id = int(np.argmax(probs))
            raw_label = model.config.id2label.get(pred_id, str(pred_id))
            label = normalize_sentiment_label(raw_label, label_map)
            score = float(probs[pred_id])
            if neutral_threshold is not None and score < neutral_threshold:
                label = neutral_label
            labels.append(label)
            scores.append(score)
    else:
        device = 0 if torch.cuda.is_available() else -1
        clf = pipeline("sentiment-analysis", model=model_name, device=device)
        for i in tqdm(range(0, len(df), batch_size)):
            batch_texts = df[text_col].iloc[i : i + batch_size].tolist()
            outputs = clf(
                batch_texts,
                truncation=True,
                max_length=int(config.get("max_length", 256)),
                padding=True,
            )
            for out in outputs:
                label = normalize_sentiment_label(out["label"], label_map)
                score = float(out["score"])
                if neutral_threshold is not None and score < neutral_threshold:
                    label = neutral_label
                labels.append(label)
                scores.append(score)

    pred_df = pd.DataFrame(
        {
            "post_id": df["post_id"].values,
            "sentiment_label": labels,
            "sentiment_score": scores,
        }
    )
    return pred_df, model_name


def train_or_pretrained_sentiment(
    df: pd.DataFrame,
    text_col: str = "text_clean",
    label_col: str = "sentiment_label",
) -> tuple[pd.DataFrame, str]:
    if df[label_col].notna().any():
        trainer, tokenizer, label2id, id2label, raw_test_df = train_text_classifier(
            df,
            text_col=text_col,
            label_col=label_col,
            model_name=SENTIMENT_FINETUNE_MODEL,
            art_dir=ART_DIR / "sentiment_clf",
            num_epochs=2,
        )

        preds = trainer.predict(trainer.eval_dataset)
        y_true = preds.label_ids
        y_pred = np.argmax(preds.predictions, axis=1)

        label_ids = list(range(len(id2label)))
        cm = confusion_matrix(y_true, y_pred, labels=label_ids)
        disp = ConfusionMatrixDisplay(
            cm, display_labels=[id2label[i] for i in label_ids]
        )
        fig, ax = plt.subplots(figsize=(6, 4))
        disp.plot(ax=ax, xticks_rotation=45)
        plt.title("Sentiment confusion matrix")
        plt.show()

        error_idx = np.where(y_true != y_pred)[0][:10]
        if len(error_idx) > 0:
            errors_df = raw_test_df.iloc[error_idx].copy()
            errors_df["pred_label"] = [id2label[i] for i in y_pred[error_idx]]
            errors_df["true_label"] = [id2label[i] for i in y_true[error_idx]]
            display(errors_df[[text_col, "true_label", "pred_label"]])

        full_ds = Dataset.from_pandas(df[[text_col]], preserve_index=False)
        tokenizer = AutoTokenizer.from_pretrained(str(ART_DIR / "sentiment_clf"))

        def tokenize(batch):
            return tokenizer(
                batch[text_col],
                truncation=True,
                padding="max_length",
                max_length=256,
            )

        full_ds = full_ds.map(tokenize, batched=True)
        full_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
        full_preds = trainer.predict(full_ds)
        probs = softmax_np(full_preds.predictions)
        pred_ids = probs.argmax(axis=1)

        pred_df = pd.DataFrame(
            {
                "post_id": df["post_id"].values,
                "sentiment_label": [id2label[i] for i in pred_ids],
                "sentiment_score": probs.max(axis=1),
            }
        )
        return pred_df, SENTIMENT_FINETUNE_MODEL

    pred_df, model_name = pretrained_sentiment(
        df,
        text_col=text_col,
        model_name=SENTIMENT_PRETRAINED_MODEL,
        config=SENTIMENT_CONFIG,
    )
    return pred_df, model_name

In [ ]:
sent_pred_df, sent_model_name = train_or_pretrained_sentiment(df)

(sent_pred_df).to_parquet(OUT_DIR / "sentiment_predictions.parquet", index=False)

if "sentiment_label_gold" not in df.columns:
    df["sentiment_label_gold"] = df["sentiment_label"]

pred_cols = sent_pred_df.rename(
    columns={"sentiment_label": "sentiment_label_pred", "sentiment_score": "sentiment_score"}
)

df = df.merge(pred_cols, on="post_id", how="left")
df["sentiment_label"] = df["sentiment_label_pred"].fillna(df["sentiment_label"])

print("Sentiment model:", sent_model_name)

display(df["sentiment_label"].value_counts().to_frame("count"))
plt.figure(figsize=(6, 4))
(df["sentiment_label"].value_counts()).plot(kind="bar", color="#9dc3e6")
plt.title("Predicted sentiment distribution")
plt.ylabel("Posts")
plt.show()

# sanity-check examples
sample_texts = df[["text_clean", "sentiment_label"]].sample(20, random_state=42)
display(sample_texts)

## E) NER (PER/ORG/LOC)

Natasha NewsNERTagger с нормализацией и EDA по сущностям.

In [ ]:
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc


def normalize_entity(ent: str) -> str:
    ent = normalize_whitespace(ent)
    ent = re.sub(r"[\"'«»]", "", ent)
    ent = re.sub(r"[\(\)\[\]{}]", "", ent)
    ent = re.sub(r"[‐‑‒–—−]", "-", ent)
    ent = re.sub(r"\s*-\s*", "-", ent)
    ent = re.sub(r"\s+", " ", ent).strip()
    if ent.isupper():
        ent = ent.title()
    return ent


def init_ner():
    segmenter = Segmenter()
    emb = NewsEmbedding()
    ner = NewsNERTagger(emb)
    return segmenter, ner


def extract_entities(text: str, segmenter, ner) -> dict:
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_ner(ner)

    buckets = {"PER": [], "ORG": [], "LOC": []}
    for span in doc.spans:
        if span.type in buckets:
            ent = normalize_entity(span.text)
            if len(ent) >= 3 and not ent.isnumeric():
                buckets[span.type].append(ent)

    # дедуп внутри документа
    for k in buckets:
        buckets[k] = list(dict.fromkeys(buckets[k]))
    return buckets

In [ ]:
import pymorphy2
from razdel import tokenize
from collections import Counter

NER_CANON_CONFIG = {
    "min_len": 3,
    "stopwords": [
        "глава",
        "президент",
        "министр",
        "депутат",
        "чиновник",
        "правительство",
        "администрация",
        "суд",
        "прокуратура",
        "город",
        "район",
        "область",
        "край",
        "республика",
        "компания",
        "корпорация",
        "банк",
        "партия",
        "комитет",
        "совет",
        "федерация",
        "страна",
        "страны",
        "улица",
    ],
    "org_legal_forms": [
        "ооо",
        "оао",
        "зао",
        "пао",
        "ао",
        "нко",
        "ип",
        "гуп",
        "муп",
        "фгуп",
        "уфгуп",
        "кфх",
    ],
    "synonyms": {
        "LOC": {
            "сша": "Соединенные Штаты",
            "соединенные штаты америки": "Соединенные Штаты",
            "рф": "Россия",
            "российская федерация": "Россия",
        },
        "ORG": {},
        "PER": {},
    },
}


def normalize_entity_text(text: str) -> str:
    text = normalize_whitespace(text)
    text = re.sub(r"[\"'«»]", "", text)
    text = re.sub(r"[\(\)\[\]{}]", "", text)
    text = re.sub(r"[‐‑‒–—−]", "-", text)
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_entity_id(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^0-9a-zа-яё\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.replace(" ", "_")


def is_noise_entity(text: str, stopwords: set[str], min_len: int) -> bool:
    if not text:
        return True
    compact = re.sub(r"[^0-9a-zа-яё]+", "", text.lower())
    if len(compact) < min_len:
        return True
    if compact.isdigit():
        return True
    if text.lower() in stopwords and len(text.split()) == 1:
        return True
    return False


def apply_synonym(text: str, ent_type: str, synonyms: dict) -> str | None:
    key = text.lower()
    if ent_type in synonyms and key in synonyms[ent_type]:
        return synonyms[ent_type][key]
    return None


def remove_org_legal_forms(text: str, legal_forms: set[str]) -> str:
    tokens = [t for t in text.split() if t]
    cleaned = [t for t in tokens if t.lower().strip(".") not in legal_forms]
    return " ".join(cleaned) if cleaned else text


def canonicalize_loc(text: str, morph, synonyms: dict) -> tuple[str, str]:
    norm = normalize_entity_text(text)
    synonym = apply_synonym(norm, "LOC", synonyms)
    if synonym:
        display = synonym
    else:
        tokens = [t.text for t in tokenize(norm)]
        lemmas = []
        for tok in tokens:
            if not re.search(r"[A-Za-zА-Яа-яЁё]", tok):
                continue
            if tok.isupper() and len(tok) <= 5:
                lemmas.append(tok)
            else:
                lemma = morph.parse(tok)[0].normal_form
                lemmas.append(lemma)
        display = " ".join([t if t.isupper() else t.title() for t in lemmas]) if lemmas else norm
    entity_id = make_entity_id(display)
    return display, entity_id


def canonicalize_org(text: str, legal_forms: set[str], synonyms: dict) -> tuple[str, str]:
    norm = normalize_entity_text(text)
    synonym = apply_synonym(norm, "ORG", synonyms)
    display = synonym or norm
    id_base = remove_org_legal_forms(display, legal_forms)
    entity_id = make_entity_id(id_base)
    return display, entity_id


def canonicalize_person(text: str, morph) -> dict | None:
    norm = normalize_entity_text(text)
    tokens = [t.text for t in tokenize(norm)]
    tokens = [t for t in tokens if re.search(r"[A-Za-zА-Яа-яЁё]", t)]
    if not tokens:
        return None
    parsed = []
    for tok in tokens:
        parses = morph.parse(tok)
        best = parses[0]
        role = None
        lemma = best.normal_form
        for p in parses:
            if "Surn" in p.tag:
                role = "Surn"
                lemma = p.normal_form
                break
            if "Name" in p.tag and role is None:
                role = "Name"
                lemma = p.normal_form
            if "Patr" in p.tag and role is None:
                role = "Patr"
                lemma = p.normal_form
        parsed.append({"lemma": lemma, "role": role})
    surname = next((p["lemma"] for p in parsed if p["role"] == "Surn"), None)
    name = next((p["lemma"] for p in parsed if p["role"] == "Name"), None)
    patronymic = next((p["lemma"] for p in parsed if p["role"] == "Patr"), None)
    lemmas = [p["lemma"] for p in parsed]

    if len(tokens) == 1:
        surname = surname or lemmas[0]

    if surname or name:
        ordered = [surname, name, patronymic]
        ordered = [w for w in ordered if w]
        display = " ".join([w.title() for w in ordered])
    else:
        display = " ".join([w.title() for w in lemmas])

    entity_id = make_entity_id(display)
    return {
        "entity_id": entity_id,
        "display": display,
        "surname": surname,
        "token_count": len(tokens),
    }


def canonicalize_entities(entities_df: pd.DataFrame, cfg: dict, art_dir: Path | None = None):
    morph = pymorphy2.MorphAnalyzer()
    stopwords = set(cfg.get("stopwords", []))
    legal_forms = set(cfg.get("org_legal_forms", []))
    synonyms = cfg.get("synonyms", {})
    min_len = cfg.get("min_len", 3)

    alias_records = []
    per_records = []
    raw_total = 0
    raw_unique = {t: set() for t in ["PER", "ORG", "LOC"]}

    for row in entities_df.itertuples(index=False):
        post_id = getattr(row, "post_id")
        for ent_type in ["PER", "ORG", "LOC"]:
            ents = getattr(row, ent_type, []) or []
            if not isinstance(ents, list):
                ents = []
            for ent in ents:
                raw_total += 1
                raw_unique[ent_type].add(str(ent))
                cleaned = normalize_entity_text(str(ent))
                if is_noise_entity(cleaned, stopwords, min_len):
                    continue

                if ent_type == "PER":
                    info = canonicalize_person(cleaned, morph)
                    if not info:
                        continue
                    per_records.append(
                        {
                            "post_id": post_id,
                            "alias": cleaned,
                            "entity_type": "PER",
                            **info,
                        }
                    )
                elif ent_type == "ORG":
                    display, entity_id = canonicalize_org(cleaned, legal_forms, synonyms)
                    alias_records.append(
                        {
                            "post_id": post_id,
                            "entity_id": entity_id,
                            "entity_type": "ORG",
                            "entity_display": display,
                            "alias": cleaned,
                        }
                    )
                else:
                    display, entity_id = canonicalize_loc(cleaned, morph, synonyms)
                    alias_records.append(
                        {
                            "post_id": post_id,
                            "entity_id": entity_id,
                            "entity_type": "LOC",
                            "entity_display": display,
                            "alias": cleaned,
                        }
                    )

    full_counts = Counter()
    full_display_by_id = {}
    for rec in per_records:
        if rec["token_count"] >= 2 and rec["surname"]:
            key = (rec["surname"], rec["entity_id"])
            full_counts[key] += 1
            if rec["entity_id"] not in full_display_by_id or len(rec["display"]) > len(
                full_display_by_id[rec["entity_id"]]
            ):
                full_display_by_id[rec["entity_id"]] = rec["display"]

    surname_to_full = {}
    for (surname, ent_id), cnt in full_counts.items():
        if surname not in surname_to_full:
            surname_to_full[surname] = {"entity_id": ent_id, "cnt": cnt}
        else:
            curr = surname_to_full[surname]
            if cnt > curr["cnt"]:
                surname_to_full[surname] = {"entity_id": ent_id, "cnt": cnt}
            elif cnt == curr["cnt"]:
                curr_disp = full_display_by_id.get(curr["entity_id"], "")
                new_disp = full_display_by_id.get(ent_id, "")
                if len(new_disp) > len(curr_disp):
                    surname_to_full[surname] = {"entity_id": ent_id, "cnt": cnt}

    for rec in per_records:
        entity_id = rec["entity_id"]
        display = rec["display"]
        if rec["token_count"] == 1 and rec["surname"] in surname_to_full:
            entity_id = surname_to_full[rec["surname"]]["entity_id"]
            display = full_display_by_id.get(entity_id, display)
        alias_records.append(
            {
                "post_id": rec["post_id"],
                "entity_id": entity_id,
                "entity_type": "PER",
                "entity_display": display,
                "alias": rec["alias"],
            }
        )

    alias_df_raw = pd.DataFrame(alias_records)
    if alias_df_raw.empty:
        empty_doc = entities_df[["post_id"]].copy()
        for ent_type in ["PER", "ORG", "LOC"]:
            empty_doc[ent_type] = [[] for _ in range(len(empty_doc))]
            empty_doc[f"{ent_type}_display"] = [[] for _ in range(len(empty_doc))]
        return (
            empty_doc,
            pd.DataFrame(columns=["entity_id", "entity_type", "entity_display", "alias"]),
            {},
            {},
        )

    display_map = (
        alias_df_raw.groupby("entity_id")["entity_display"]
        .agg(lambda s: max(Counter(s).items(), key=lambda x: (x[1], len(x[0])))[0])
        .to_dict()
    )

    alias_df = alias_df_raw.drop(columns=["post_id"]).drop_duplicates()
    alias_df["entity_display"] = alias_df["entity_id"].map(display_map)

    doc_entities = entities_df[["post_id"]].copy()
    for ent_type in ["PER", "ORG", "LOC"]:
        subset = alias_df_raw[alias_df_raw["entity_type"] == ent_type]
        grouped_ids = (
            subset.groupby("post_id")["entity_id"]
            .apply(lambda x: list(dict.fromkeys(x)))
            .to_dict()
        )
        grouped_disp = (
            subset.groupby("post_id")["entity_id"]
            .apply(lambda ids: [display_map.get(i, i) for i in dict.fromkeys(ids)])
            .to_dict()
        )
        doc_entities[ent_type] = doc_entities["post_id"].map(grouped_ids).apply(
            lambda x: x if isinstance(x, list) else []
        )
        doc_entities[f"{ent_type}_display"] = doc_entities["post_id"].map(
            grouped_disp
        ).apply(lambda x: x if isinstance(x, list) else [])

    canon_unique = {
        t: len(set([e for row in doc_entities[t] for e in row])) for t in ["PER", "ORG", "LOC"]
    }
    alias_counts = (
        alias_df.groupby("entity_id")["alias"].nunique().sort_values(ascending=False)
    )

    quality = {
        "raw_total": raw_total,
        "kept_total": int(alias_df_raw.shape[0]),
        "noise_ratio": 1 - (alias_df_raw.shape[0] / raw_total) if raw_total else 0.0,
        "raw_unique": {k: len(v) for k, v in raw_unique.items()},
        "canon_unique": canon_unique,
        "top_alias_counts": alias_counts.head(10).to_dict(),
    }

    if art_dir is not None:
        art_dir = Path(art_dir)
        art_dir.mkdir(parents=True, exist_ok=True)
        with open(art_dir / "canonicalization_config.json", "w", encoding="utf-8") as f:
            json.dump(cfg, f, ensure_ascii=False, indent=2)
        meta = {
            "pymorphy2_version": getattr(pymorphy2, "__version__", "unknown"),
            "natasha_version": getattr(__import__("natasha"), "__version__", "unknown"),
        }
        with open(art_dir / "canonicalization_meta.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

    return doc_entities, alias_df, display_map, quality


segmenter, ner = init_ner()

entities_raw = []
for text in tqdm(df["text_clean"].tolist()):
    entities_raw.append(extract_entities(text, segmenter, ner))

entities_raw_df = pd.DataFrame(entities_raw)
entities_raw_df["post_id"] = df["post_id"].values

entities_raw_df.to_parquet(OUT_DIR / "doc_entities_raw.parquet", index=False)

entities_df, entity_aliases_df, entity_display_map, ner_quality = canonicalize_entities(
    entities_raw_df,
    NER_CANON_CONFIG,
    art_dir=ART_DIR / "ner_canon",
)

entities_df.to_parquet(OUT_DIR / "doc_entities.parquet", index=False)
entity_aliases_df.to_parquet(OUT_DIR / "entity_aliases.parquet", index=False)

print("NER canonicalization quality:")
print("Raw total:", ner_quality.get("raw_total"))
print("Kept total:", ner_quality.get("kept_total"))
print("Noise ratio:", round(ner_quality.get("noise_ratio", 0.0), 3))
print("Unique raw:", ner_quality.get("raw_unique"))
print("Unique canonical:", ner_quality.get("canon_unique"))
display(pd.Series(ner_quality.get("top_alias_counts", {})).to_frame("alias_count"))

entities_by_id = entities_df.set_index("post_id")


def _display_top_entities(entities_df, ent_type, display_map):
    flat = [e for row in entities_df[ent_type] for e in row]
    top = pd.Series(flat).value_counts().head(20)
    top.index = [display_map.get(x, x) for x in top.index]
    display(top.to_frame("count").rename_axis(ent_type))
    plt.figure(figsize=(8, 4))
    top.plot(kind="bar", color="#c55a11")
    plt.title(f"Top {ent_type} entities (canonical)")
    plt.ylabel("Count")
    plt.show()


# EDA: top entities
for ent_type in ["PER", "ORG", "LOC"]:
    _display_top_entities(entities_df, ent_type, entity_display_map)

# Top entities by topic (если есть)
if df["topic_label"].notna().any():
    merged = df[["post_id", "topic_label"]].merge(entities_df, on="post_id")
    for ent_type in ["PER", "ORG", "LOC"]:
        print(f"\nTop {ent_type} by topic:")
        for topic, group in merged.groupby("topic_label"):
            flat = [e for row in group[ent_type] for e in row]
            top = pd.Series(flat).value_counts().head(5)
            names = [entity_display_map.get(x, x) for x in top.index.tolist()]
            print(topic, ":", ", ".join(names))

# Demo checks for common merges (if present)
demo_aliases = {
    "LOC": ["киев", "киеве", "киева"],
    "PER": ["путин", "владимир путин"],
}
for ent_type, variants in demo_aliases.items():
    subset = entity_aliases_df[
        (entity_aliases_df["entity_type"] == ent_type)
        & (entity_aliases_df["alias"].str.lower().isin(variants))
    ]
    if subset.empty:
        print(f"Demo merge for {ent_type}: not found in sample")
        continue
    grouped = subset.groupby("entity_id")["entity_display"].first()
    print(f"Demo merge for {ent_type}:", ", ".join(grouped.tolist()))

## F) Embeddings + Theme Clustering

SBERT эмбеддинги + UMAP → HDBSCAN с pre-bucketing (topic × time window).

In [ ]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer

SBERT_MODEL = "ai-forever/sbert_large_nlu_ru"


def compute_embeddings(
    texts: list[str],
    model_name: str = SBERT_MODEL,
    batch_size: int = 64,
    normalize_embeddings: bool = True,
) -> np.ndarray:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=normalize_embeddings,
    )

    embedder_dir = ART_DIR / "embedder"
    embedder_dir.mkdir(parents=True, exist_ok=True)
    model.save(str(embedder_dir))
    with open(embedder_dir / "embedder_meta.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "model_name": model_name,
                "normalize_embeddings": normalize_embeddings,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    return embeddings

In [ ]:
RU_STOPWORDS = {
    "и",
    "в",
    "во",
    "не",
    "что",
    "он",
    "на",
    "я",
    "с",
    "со",
    "как",
    "а",
    "то",
    "все",
    "она",
    "так",
    "его",
    "но",
    "да",
    "ты",
    "к",
    "у",
    "же",
    "вы",
    "за",
    "бы",
    "по",
    "только",
    "ее",
    "мне",
    "было",
    "вот",
    "от",
    "меня",
    "еще",
    "нет",
    "о",
    "из",
    "ему",
    "теперь",
    "когда",
    "даже",
    "ну",
    "вдруг",
    "ли",
    "если",
    "уже",
    "или",
    "ни",
    "быть",
    "был",
    "него",
    "до",
    "вас",
    "нибудь",
    "опять",
    "уж",
    "вам",
    "ведь",
    "там",
    "потом",
    "себя",
    "ничего",
    "ей",
    "может",
    "они",
    "тут",
    "где",
    "есть",
    "надо",
    "ней",
    "для",
    "мы",
    "тебя",
    "их",
    "чем",
    "была",
    "сам",
    "чтоб",
    "без",
    "будто",
    "чего",
    "раз",
    "тоже",
    "себе",
    "под",
    "будет",
    "ж",
    "тогда",
    "кто",
    "этот",
    "того",
    "потому",
    "этого",
    "какой",
    "совсем",
    "ним",
    "здесь",
    "этом",
    "один",
    "почти",
    "мой",
    "тем",
    "чтобы",
    "нее",
    "кажется",
    "сейчас",
    "были",
    "куда",
    "зачем",
    "всех",
    "никогда",
    "можно",
    "при",
    "наконец",
    "два",
    "об",
    "другой",
    "хоть",
    "после",
    "над",
    "больше",
    "тот",
    "через",
    "эти",
    "нас",
    "про",
    "них",
    "какая",
    "много",
    "разве",
    "три",
    "эту",
    "моя",
    "впрочем",
    "хорошо",
    "свою",
    "этой",
    "перед",
    "иногда",
    "лучше",
    "чуть",
    "том",
    "нельзя",
    "такой",
    "им",
    "более",
    "всегда",
    "конечно",
    "всю",
    "между",
}

DOMAIN_STOPWORDS = {
    "сегодня",
    "сообщил",
    "сообщила",
    "сообщает",
    "заявил",
    "заявила",
    "заявили",
    "заявляет",
    "сказал",
    "сказала",
    "отметил",
    "отметила",
    "telegram",
    "телеграм",
    "канал",
    "пост",
}


def make_buckets(
    df: pd.DataFrame,
    topic_col: str = "topic_label",
    time_col: str = "date",
    window_hours: int = 6,
) -> pd.DataFrame:
    df = df.copy()

    if df[time_col].notna().any():
        df["time_bucket"] = df[time_col].dt.floor(f"{window_hours}H")
    else:
        df["time_bucket"] = pd.NaT

    if topic_col in df.columns and df[topic_col].notna().any():
        df["topic_bucket"] = df[topic_col].fillna("unknown")
    else:
        df["topic_bucket"] = "unknown"

    df["bucket_id"] = df["topic_bucket"].astype(str) + "|" + df["time_bucket"].astype(str)
    return df


def cluster_by_bucket(
    df: pd.DataFrame,
    embeddings: np.ndarray,
    bucket_col: str,
    cfg: dict,
    store_models: bool = False,
) -> tuple[np.ndarray, np.ndarray, dict]:
    labels = np.full(len(df), -1, dtype=int)
    probs = np.zeros(len(df), dtype=float)
    bucket_models = {}

    cluster_offset = 0
    for bucket_id, idx in df.groupby(bucket_col).indices.items():
        idx = np.array(list(idx))
        if len(idx) < cfg["min_cluster_size"]:
            continue

        umap_model = umap.UMAP(
            n_components=50,
            metric="cosine",
            n_neighbors=cfg["n_neighbors"],
            min_dist=cfg["min_dist"],
            random_state=42,
        )
        reduced = umap_model.fit_transform(embeddings[idx])

        hdb = hdbscan.HDBSCAN(
            min_cluster_size=cfg["min_cluster_size"],
            min_samples=cfg["min_samples"],
            cluster_selection_method="leaf",
            allow_single_cluster=False,
            prediction_data=True,
        )
        hdb.fit(reduced)

        bucket_labels = hdb.labels_
        bucket_probs = hdb.probabilities_

        n_clusters = len(set(bucket_labels)) - (1 if -1 in set(bucket_labels) else 0)
        if n_clusters > 0:
            mapped = np.where(bucket_labels == -1, -1, bucket_labels + cluster_offset)
            cluster_offset += n_clusters
        else:
            mapped = bucket_labels

        labels[idx] = mapped
        probs[idx] = bucket_probs

        if store_models:
            bucket_models[bucket_id] = {"umap": umap_model, "hdbscan": hdb}

    return labels, probs, bucket_models


def evaluate_configs(
    df: pd.DataFrame,
    embeddings: np.ndarray,
    bucket_col: str,
    configs: list[dict],
) -> pd.DataFrame:
    rows = []
    for cfg_id, cfg in enumerate(configs):
        labels, probs, _ = cluster_by_bucket(df, embeddings, bucket_col, cfg, store_models=False)
        noise_ratio = float((labels == -1).mean())
        valid = labels[labels >= 0]
        n_clusters = int(len(np.unique(valid))) if len(valid) > 0 else 0
        largest_cluster_ratio = 0.0
        if len(valid) > 0:
            largest_cluster_ratio = float(np.bincount(valid).max() / len(labels))

        rows.append(
            {
                "config_id": cfg_id,
                "n_neighbors": cfg["n_neighbors"],
                "min_dist": cfg["min_dist"],
                "min_cluster_size": cfg["min_cluster_size"],
                "min_samples": cfg["min_samples"],
                "noise_ratio": noise_ratio,
                "n_clusters": n_clusters,
                "largest_cluster_ratio": largest_cluster_ratio,
            }
        )
    return pd.DataFrame(rows)


def select_best_config(metrics_df: pd.DataFrame) -> pd.Series:
    def heuristic_score(row):
        score = 0.0
        if row["largest_cluster_ratio"] <= 0.35:
            score += 1.0
        if 0.05 < row["noise_ratio"] < 0.9:
            score += 1.0
        if row["n_clusters"] > 2:
            score += 0.5
        score += min(row["n_clusters"] / 20.0, 1.0)
        score -= row["largest_cluster_ratio"]
        return score

    metrics_df = metrics_df.copy()
    metrics_df["heuristic_score"] = metrics_df.apply(heuristic_score, axis=1)
    best_row = metrics_df.sort_values("heuristic_score", ascending=False).iloc[0]
    return best_row


def label_clusters(
    df: pd.DataFrame,
    cluster_col: str,
    text_col: str,
    top_n_terms: int = 10,
) -> tuple[pd.DataFrame, TfidfVectorizer]:
    cluster_docs = (
        df[df[cluster_col] >= 0]
        .groupby(cluster_col)[text_col]
        .apply(lambda texts: " ".join(texts))
    )

    vectorizer = TfidfVectorizer(
        stop_words=RU_STOPWORDS | DOMAIN_STOPWORDS,
        max_features=5000,
        ngram_range=(1, 2),
    )
    X = vectorizer.fit_transform(cluster_docs)
    terms = np.array(vectorizer.get_feature_names_out())

    rows = []
    for i, cluster_id in enumerate(cluster_docs.index):
        row = X[i].toarray().ravel()
        top_idx = row.argsort()[::-1][:top_n_terms]
        rows.append(
            {
                "cluster_id": int(cluster_id),
                "top_terms": ", ".join(terms[top_idx]),
            }
        )

    return pd.DataFrame(rows), vectorizer


def cluster_themes(
    df: pd.DataFrame,
    embeddings: np.ndarray,
    window_hours: int,
    configs: list[dict],
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, dict, pd.DataFrame, dict, pd.Series]:
    bucketed_df = make_buckets(
        df, topic_col="topic_label", time_col="date", window_hours=window_hours
    )
    metrics_df = evaluate_configs(bucketed_df, embeddings, "bucket_id", configs)
    best_cfg_row = select_best_config(metrics_df)
    best_cfg = configs[int(best_cfg_row["config_id"])]
    labels, probs, bucket_models = cluster_by_bucket(
        bucketed_df, embeddings, "bucket_id", best_cfg, store_models=True
    )
    return bucketed_df, labels, probs, bucket_models, metrics_df, best_cfg, best_cfg_row

In [ ]:
SAVE_EMBEDDINGS = False
EMB_PATH = OUT_DIR / "embeddings.npy"

texts = df["text_clean"].tolist()

if SAVE_EMBEDDINGS and EMB_PATH.exists():
    embeddings = np.load(EMB_PATH)
else:
    embeddings = compute_embeddings(texts)
    if SAVE_EMBEDDINGS:
        np.save(EMB_PATH, embeddings)

# pre-bucketing: topic × time window
WINDOW_HOURS = 6  # для событий; для "тем дня" можно поставить 24

CONFIGS = [
    {"n_neighbors": 10, "min_dist": 0.0, "min_cluster_size": 15, "min_samples": 5},
    {"n_neighbors": 10, "min_dist": 0.05, "min_cluster_size": 25, "min_samples": 5},
    {"n_neighbors": 15, "min_dist": 0.05, "min_cluster_size": 25, "min_samples": 10},
    {"n_neighbors": 15, "min_dist": 0.1, "min_cluster_size": 25, "min_samples": 10},
    {"n_neighbors": 30, "min_dist": 0.0, "min_cluster_size": 50, "min_samples": 10},
    {"n_neighbors": 30, "min_dist": 0.1, "min_cluster_size": 50, "min_samples": 15},
]

bucketed_df, labels, probs, bucket_models, metrics_df, best_cfg, best_cfg_row = cluster_themes(
    df, embeddings, WINDOW_HOURS, CONFIGS
)

plt.figure(figsize=(10, 4))
metrics_df.set_index("config_id")[["noise_ratio", "largest_cluster_ratio"]].plot(kind="bar")
plt.title("Clustering metrics by config")
plt.ylabel("Ratio")
plt.show()

plt.figure(figsize=(10, 4))
metrics_df.set_index("config_id")["n_clusters"].plot(kind="bar", color="#70ad47")
plt.title("Number of clusters by config")
plt.ylabel("Clusters")
plt.show()

print("Best config:", best_cfg)

df["cluster_id"] = labels
df["cluster_prob"] = probs

total_noise_ratio = float((df["cluster_id"] == -1).mean())
valid = df[df["cluster_id"] >= 0]["cluster_id"].values
n_clusters = int(len(np.unique(valid))) if len(valid) > 0 else 0
largest_cluster_ratio = 0.0
if len(valid) > 0:
    largest_cluster_ratio = float(np.bincount(valid).max() / len(df))

print("Noise ratio:", total_noise_ratio)
print("Number of clusters:", n_clusters)
print("Largest cluster ratio:", largest_cluster_ratio)

# cluster size distribution
sizes = df[df["cluster_id"] >= 0]["cluster_id"].value_counts()
plt.figure(figsize=(8, 4))
sizes.plot(kind="hist", bins=40, color="#5b9bd5")
plt.title("Cluster size distribution")
plt.xlabel("Cluster size")
plt.ylabel("Count")
plt.show()

# UMAP 2D scatter (sanity check)
SAMPLE_FOR_UMAP = min(len(df), 5000)
idx = np.random.choice(len(df), size=SAMPLE_FOR_UMAP, replace=False)
emb_2d = umap.UMAP(n_components=2, metric="cosine", random_state=42).fit_transform(
    embeddings[idx]
)
plt.figure(figsize=(7, 5))
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=df["cluster_id"].iloc[idx], s=5, cmap="tab20")
plt.title("UMAP 2D scatter by cluster")
plt.show()

# label clusters with c-TF-IDF style
cluster_labels_df, vectorizer = label_clusters(df, "cluster_id", "text_clean", top_n_terms=10)

TOP_CLUSTERS = 10
print("\nTop clusters summary:")
for cluster_id in sizes.head(TOP_CLUSTERS).index:
    terms = cluster_labels_df.loc[
        cluster_labels_df["cluster_id"] == cluster_id, "top_terms"
    ].values
    terms_str = terms[0] if len(terms) else ""
    cluster_texts = df[df["cluster_id"] == cluster_id]["text_clean"].head(5).tolist()

    ents = entities_df.loc[df["cluster_id"] == cluster_id, ["PER", "ORG", "LOC"]]
    ent_flat = []
    for _, row in ents.iterrows():
        ent_flat.extend(row["PER"] + row["ORG"] + row["LOC"])
    top_ents = pd.Series(ent_flat).value_counts().head(5).index.tolist()

    print(f"\nCluster {cluster_id}")
    print("Top terms:", terms_str)
    print("Top entities:", ", ".join(top_ents))
    print("Examples:")
    for t in cluster_texts:
        print("-", t[:200])

# save clustering artifacts
cluster_art_dir = ART_DIR / "clustering"
cluster_art_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "config": best_cfg,
        "bucket_models": bucket_models,
        "vectorizer": vectorizer,
        "window_hours": WINDOW_HOURS,
    },
    cluster_art_dir / "umap_hdbscan.joblib",
)

clusters_out = df[["post_id", "cluster_id", "cluster_prob", "topic_label", "date"]].copy()
clusters_out["bucket_id"] = bucketed_df["bucket_id"].values
clusters_out["noise_ratio"] = total_noise_ratio
clusters_out["n_clusters"] = n_clusters
clusters_out["largest_cluster_ratio"] = largest_cluster_ratio
clusters_out.to_parquet(OUT_DIR / "clusters.parquet", index=False)

metrics_df.to_csv(OUT_DIR / "cluster_config_metrics.csv", index=False)

## G) Same-theme / pairwise scoring

Минимальный скоринг для пары документов: cosine + entity overlap + time penalty.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def time_penalty(ts_a, ts_b, tau_hours: float = 24.0) -> float:
    if pd.isna(ts_a) or pd.isna(ts_b):
        return 1.0
    delta_h = abs((ts_a - ts_b).total_seconds()) / 3600.0
    return math.exp(-delta_h / tau_hours)


def same_theme_score(row_a, row_b, emb_a, emb_b) -> dict:
    cosine = float(np.dot(emb_a, emb_b))
    ents_a = set(row_a["PER"] + row_a["ORG"] + row_a["LOC"])
    ents_b = set(row_b["PER"] + row_b["ORG"] + row_b["LOC"])
    ent_overlap = jaccard(ents_a, ents_b)
    time_score = time_penalty(row_a["date"], row_b["date"])

    score = 0.6 * cosine + 0.25 * ent_overlap + 0.15 * time_score
    return {
        "score": score,
        "cosine": cosine,
        "entity_jaccard": ent_overlap,
        "time_penalty": time_score,
    }


df_entities = df.merge(entities_df, on="post_id")

PAIR_SAMPLE = min(len(df_entities), 800)
if PAIR_SAMPLE >= 2:
    idx = np.random.choice(len(df_entities), size=PAIR_SAMPLE, replace=False)
    sample_df = df_entities.iloc[idx].reset_index(drop=True)
    sample_emb = embeddings[idx]
    sim_matrix = cosine_similarity(sample_emb)

    # top similar pairs
    triu_idx = np.triu_indices_from(sim_matrix, k=1)
    sims = sim_matrix[triu_idx]
    top_k = 10
    top_pairs_idx = sims.argsort()[::-1][:top_k]

    pairs = []
    for rank_idx in top_pairs_idx:
        i = triu_idx[0][rank_idx]
        j = triu_idx[1][rank_idx]
        score = same_theme_score(
            sample_df.loc[i],
            sample_df.loc[j],
            sample_emb[i],
            sample_emb[j],
        )
        pairs.append(
            {
                "pair_type": "top",
                "post_id_a": sample_df.loc[i, "post_id"],
                "post_id_b": sample_df.loc[j, "post_id"],
                **score,
            }
        )

    # borderline pairs
    border_mask = (sims >= 0.5) & (sims <= 0.6)
    border_idx = np.where(border_mask)[0][:10]
    for rank_idx in border_idx:
        i = triu_idx[0][rank_idx]
        j = triu_idx[1][rank_idx]
        score = same_theme_score(
            sample_df.loc[i],
            sample_df.loc[j],
            sample_emb[i],
            sample_emb[j],
        )
        pairs.append(
            {
                "pair_type": "borderline",
                "post_id_a": sample_df.loc[i, "post_id"],
                "post_id_b": sample_df.loc[j, "post_id"],
                **score,
            }
        )

    # random pairs
    rng = np.random.default_rng(42)
    for _ in range(10):
        i, j = rng.choice(PAIR_SAMPLE, size=2, replace=False)
        score = same_theme_score(
            sample_df.loc[i],
            sample_df.loc[j],
            sample_emb[i],
            sample_emb[j],
        )
        pairs.append(
            {
                "pair_type": "random",
                "post_id_a": sample_df.loc[i, "post_id"],
                "post_id_b": sample_df.loc[j, "post_id"],
                **score,
            }
        )

    pairwise_df = pd.DataFrame(pairs)
    pairwise_df.to_csv(OUT_DIR / "pairwise_examples.csv", index=False)
    display(pairwise_df.head(20))
else:
    print("Not enough documents for pairwise scoring.")

## H) Graph for UI

Граф theme ↔ entities ↔ category (topic) с визуализацией в pyvis.

In [ ]:
import networkx as nx
from pyvis.network import Network
from itertools import combinations
from collections import Counter

GRAPH_CONFIG = {
    "max_themes": 30,
    "max_entities": 300,
    "min_edge_weight": 3,
    "max_neighbors": 20,
    "co_mention_min": 3,
    "co_mention_pmi_min": 0.0,
    "time_window_hours": None,
    "include_msg_nodes": False,
    "entity_stoplist": [],
    "use_idf": True,
}


def _make_node_id(prefix: str, value: str) -> str:
    return f"{prefix}:{value}"


def _limit_top_neighbors(edges_df: pd.DataFrame, max_neighbors: int) -> pd.DataFrame:
    if not max_neighbors or edges_df.empty:
        return edges_df
    df = edges_df.copy()
    df["rank_src"] = df.groupby("src")["weight"].rank("first", ascending=False)
    df = df[df["rank_src"] <= max_neighbors]
    df["rank_dst"] = df.groupby("dst")["weight"].rank("first", ascending=False)
    df = df[df["rank_dst"] <= max_neighbors]
    return df.drop(columns=["rank_src", "rank_dst"])


def build_graph_nodes_edges(
    docs_df: pd.DataFrame,
    entities_df: pd.DataFrame,
    alias_df: pd.DataFrame,
    config: dict = GRAPH_CONFIG,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    use_clusters = "cluster_id" in docs_df.columns and (docs_df["cluster_id"] >= 0).any()
    theme_col = "cluster_id" if use_clusters else "topic_label"
    theme_type = "CLUSTER" if use_clusters else "TOPIC"

    df = docs_df.copy()
    if use_clusters:
        df = df[df["cluster_id"] >= 0].copy()
    else:
        df = df[df["topic_label"].notna()].copy()

    if df.empty:
        return (
            pd.DataFrame(columns=["node_id", "node_type", "label", "attrs"]),
            pd.DataFrame(columns=["src", "dst", "edge_type", "weight", "attrs"]),
            pd.DataFrame(columns=["node_id", "degree", "weighted_degree"]),
        )

    theme_counts = df[theme_col].value_counts().head(config["max_themes"])
    df = df[df[theme_col].isin(theme_counts.index)].copy()

    merged = df[["post_id", "channel", "date", theme_col]].merge(entities_df, on="post_id")
    for col in ["PER", "ORG", "LOC"]:
        merged[col] = merged[col].apply(lambda x: x if isinstance(x, list) else [])

    merged["entities"] = merged.apply(
        lambda r: list(dict.fromkeys(r["PER"] + r["ORG"] + r["LOC"])), axis=1
    )

    entity_stoplist = {make_entity_id(x) for x in config.get("entity_stoplist", [])}

    doc_entity_counts = Counter()
    for ents in merged["entities"]:
        unique = set(e for e in ents if e not in entity_stoplist)
        for e in unique:
            doc_entity_counts[e] += 1

    top_entities = {
        e for e, _ in doc_entity_counts.most_common(config["max_entities"])
    }

    def theme_node(value) -> str:
        if theme_type == "CLUSTER":
            return _make_node_id("cluster", str(int(value)))
        return _make_node_id("topic", make_entity_id(str(value)))

    def channel_node(value) -> str:
        return _make_node_id("channel", make_entity_id(str(value)))

    def entity_node(value) -> str:
        return _make_node_id("entity", value)

    merged["theme_id"] = merged[theme_col].apply(theme_node)
    merged["channel_id"] = merged["channel"].apply(channel_node)
    merged["entities"] = merged["entities"].apply(
        lambda xs: [x for x in xs if x in top_entities and x not in entity_stoplist]
    )

    if config.get("time_window_hours") and merged["date"].notna().any():
        merged["time_bucket"] = merged["date"].dt.floor(
            f"{int(config['time_window_hours'])}H"
        )
    else:
        merged["time_bucket"] = pd.NaT

    entity_meta = (
        alias_df.groupby("entity_id")
        .agg(
            entity_display=("entity_display", "first"),
            entity_type=("entity_type", "first"),
            alias_count=("alias", "nunique"),
        )
        .reset_index()
    )
    entity_meta = entity_meta[entity_meta["entity_id"].isin(top_entities)]

    nodes = []
    for _, row in entity_meta.iterrows():
        nodes.append(
            {
                "node_id": entity_node(row["entity_id"]),
                "node_type": "ENTITY",
                "label": row["entity_display"],
                "attrs": json.dumps(
                    {
                        "entity_type": row["entity_type"],
                        "alias_count": int(row["alias_count"]),
                        "doc_freq": int(doc_entity_counts.get(row["entity_id"], 0)),
                    },
                    ensure_ascii=False,
                ),
            }
        )

    for theme_value, count in theme_counts.items():
        nodes.append(
            {
                "node_id": theme_node(theme_value),
                "node_type": theme_type,
                "label": str(theme_value),
                "attrs": json.dumps({"doc_count": int(count)}, ensure_ascii=False),
            }
        )

    for channel, count in df["channel"].value_counts().items():
        nodes.append(
            {
                "node_id": channel_node(channel),
                "node_type": "CHANNEL",
                "label": str(channel),
                "attrs": json.dumps({"doc_count": int(count)}, ensure_ascii=False),
            }
        )

    if config.get("include_msg_nodes"):
        for post_id in merged["post_id"].unique():
            nodes.append(
                {
                    "node_id": _make_node_id("msg", str(post_id)),
                    "node_type": "MSG",
                    "label": str(post_id),
                    "attrs": json.dumps({}, ensure_ascii=False),
                }
            )

    edges_frames = []

    channel_theme = (
        merged.groupby(["channel_id", "theme_id"]).size().reset_index(name="count")
    )
    channel_theme["edge_type"] = "PUBLISHES"
    channel_theme["weight"] = channel_theme["count"]
    channel_theme["attrs"] = channel_theme["count"].apply(
        lambda c: json.dumps({"count": int(c)}, ensure_ascii=False)
    )
    channel_theme = channel_theme.rename(columns={"channel_id": "src", "theme_id": "dst"})
    edges_frames.append(channel_theme[["src", "dst", "edge_type", "weight", "attrs"]])

    theme_entity_counts = Counter()
    for _, row in merged.iterrows():
        theme_id = row["theme_id"]
        for ent in row["entities"]:
            theme_entity_counts[(theme_id, ent)] += 1

    theme_entity_df = pd.DataFrame(
        [
            {"theme_id": k[0], "entity_id": k[1], "count": v}
            for k, v in theme_entity_counts.items()
        ]
    )
    if not theme_entity_df.empty:
        theme_entity_df = theme_entity_df.merge(
            theme_entity_df.groupby("entity_id")["theme_id"].nunique().rename("theme_df"),
            on="entity_id",
        )
        num_themes = theme_counts.shape[0]
        theme_entity_df["idf"] = (
            np.log((1 + num_themes) / (1 + theme_entity_df["theme_df"])) + 1.0
            if config.get("use_idf", True)
            else 1.0
        )
        theme_entity_df["weight"] = theme_entity_df["count"] * theme_entity_df["idf"]
        theme_entity_df = theme_entity_df[theme_entity_df["weight"] >= config["min_edge_weight"]]
        theme_entity_df["edge_type"] = "MENTIONS_IN_THEME"
        theme_entity_df["attrs"] = theme_entity_df.apply(
            lambda r: json.dumps({"count": int(r["count"]), "idf": float(r["idf"])}, ensure_ascii=False),
            axis=1,
        )
        theme_entity_df = theme_entity_df.rename(
            columns={"theme_id": "src", "entity_id": "dst"}
        )
        theme_entity_df["dst"] = theme_entity_df["dst"].apply(entity_node)
        theme_entity_df = _limit_top_neighbors(theme_entity_df, config["max_neighbors"])
        edges_frames.append(
            theme_entity_df[["src", "dst", "edge_type", "weight", "attrs"]]
        )

    co_counts = Counter()
    group_cols = ["theme_id"]
    if config.get("time_window_hours") and merged["time_bucket"].notna().any():
        group_cols.append("time_bucket")

    for _, group in merged.groupby(group_cols):
        for ents in group["entities"]:
            uniq = sorted(set(ents))
            for a, b in combinations(uniq, 2):
                key = (a, b) if a < b else (b, a)
                co_counts[key] += 1

    if co_counts:
        total_docs = max(len(merged), 1)
        co_rows = []
        for (a, b), count in co_counts.items():
            if count < config["co_mention_min"]:
                continue
            pmi = math.log((count * total_docs + 1) / ((doc_entity_counts[a] * doc_entity_counts[b]) + 1))
            if pmi < config["co_mention_pmi_min"]:
                continue
            co_rows.append(
                {
                    "src": entity_node(a),
                    "dst": entity_node(b),
                    "edge_type": "CO_MENTIONS",
                    "weight": count,
                    "attrs": json.dumps({"count": int(count), "pmi": float(pmi)}, ensure_ascii=False),
                }
            )
        co_df = pd.DataFrame(co_rows)
        if not co_df.empty:
            co_df = _limit_top_neighbors(co_df, config["max_neighbors"])
            edges_frames.append(co_df)

    if config.get("include_msg_nodes"):
        msg_edges = []
        for _, row in merged.iterrows():
            msg_id = _make_node_id("msg", str(row["post_id"]))
            msg_edges.append(
                {
                    "src": row["channel_id"],
                    "dst": msg_id,
                    "edge_type": "PUBLISHED",
                    "weight": 1,
                    "attrs": json.dumps({"count": 1}, ensure_ascii=False),
                }
            )
            msg_edges.append(
                {
                    "src": msg_id,
                    "dst": row["theme_id"],
                    "edge_type": "IN_THEME",
                    "weight": 1,
                    "attrs": json.dumps({"count": 1}, ensure_ascii=False),
                }
            )
            for ent in row["entities"]:
                msg_edges.append(
                    {
                        "src": msg_id,
                        "dst": entity_node(ent),
                        "edge_type": "MENTIONS",
                        "weight": 1,
                        "attrs": json.dumps({"count": 1}, ensure_ascii=False),
                    }
                )
        edges_frames.append(pd.DataFrame(msg_edges))

    edges_df = pd.concat(edges_frames, ignore_index=True) if edges_frames else pd.DataFrame(
        columns=["src", "dst", "edge_type", "weight", "attrs"]
    )

    nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=["node_id"])

    G = nx.Graph()
    for _, row in nodes_df.iterrows():
        G.add_node(row["node_id"], node_type=row["node_type"], label=row["label"])
    for _, row in edges_df.iterrows():
        G.add_edge(row["src"], row["dst"], weight=row["weight"], edge_type=row["edge_type"])

    metrics_df = pd.DataFrame(
        {
            "node_id": list(G.nodes()),
            "degree": [G.degree(n) for n in G.nodes()],
            "weighted_degree": [G.degree(n, weight="weight") for n in G.nodes()],
        }
    )

    return nodes_df, edges_df, metrics_df


def export_graph(
    nodes_df: pd.DataFrame,
    edges_df: pd.DataFrame,
    out_dir: Path,
    html_name: str = "theme_entity_graph.html",
    metrics_df: pd.DataFrame | None = None,
) -> None:
    nodes_df.to_parquet(out_dir / "nodes.parquet", index=False)
    edges_df.to_parquet(out_dir / "edges.parquet", index=False)
    if metrics_df is not None and not metrics_df.empty:
        metrics_df.to_parquet(out_dir / "graph_metrics.parquet", index=False)

    net = Network(height="750px", width="100%", bgcolor="#ffffff", font_color="#222")
    for _, row in nodes_df.iterrows():
        attrs = json.loads(row["attrs"]) if row["attrs"] else {}
        title = "<br>".join([f"{k}: {v}" for k, v in attrs.items()])
        net.add_node(
            row["node_id"],
            label=row["label"],
            title=title,
            group=row["node_type"],
        )

    for _, row in edges_df.iterrows():
        attrs = json.loads(row["attrs"]) if row["attrs"] else {}
        title = f"{row['edge_type']} | weight={row['weight']}"
        if attrs:
            title = title + "<br>" + "<br>".join([f"{k}: {v}" for k, v in attrs.items()])
        net.add_edge(
            row["src"],
            row["dst"],
            value=row["weight"],
            label=row["edge_type"],
            title=title,
        )

    net.save_graph(str(out_dir / html_name))
    print("Graph saved to", out_dir / html_name)


nodes_df, edges_df, metrics_df = build_graph_nodes_edges(
    df,
    entities_df,
    entity_aliases_df,
    config=GRAPH_CONFIG,
)
export_graph(nodes_df, edges_df, OUT_DIR, html_name="theme_entity_graph.html", metrics_df=metrics_df)

# persist graph config
with open(ART_DIR / "graph_config.json", "w", encoding="utf-8") as f:
    json.dump(GRAPH_CONFIG, f, ensure_ascii=False, indent=2)

## I) Final “UI-ready” table

Единый датафрейм для UI/сервиса.

In [ ]:
def format_entities(row: pd.Series) -> str:
    if "PER_display" in row and "ORG_display" in row and "LOC_display" in row:
        ents = row["PER_display"] + row["ORG_display"] + row["LOC_display"]
    else:
        ents = row["PER"] + row["ORG"] + row["LOC"]
    if not ents:
        return ""
    return ", ".join(pd.Series(ents).value_counts().index.tolist()[:10])

entity_merge = entities_df.merge(df[["post_id"]], on="post_id")
entity_merge["top_entities"] = entity_merge.apply(format_entities, axis=1)

# predictions table
pred_df = df[[
    "post_id",
    "topic_label",
    "topic_score",
    "sentiment_label",
    "sentiment_score",
    "cluster_id",
    "cluster_prob",
]].copy()

pred_df["topic_model"] = topic_model_name
pred_df["sentiment_model"] = sent_model_name
pred_df["embedding_model"] = SBERT_MODEL
pred_df["clustering_config"] = json.dumps(best_cfg)

pred_df.to_parquet(OUT_DIR / "doc_predictions.parquet", index=False)

# final UI-ready table
final_df = df[["post_id", "date", "channel", "text_clean"]].copy()
final_df["text_snippet"] = final_df["text_clean"].str.slice(0, 220)
final_df = final_df.drop(columns=["text_clean"]).merge(
    pred_df,
    on="post_id",
    how="left",
).merge(
    entity_merge[["post_id", "top_entities"]],
    on="post_id",
    how="left",
)

final_df.to_parquet(OUT_DIR / "final_table.parquet", index=False)

print("Saved final table to", OUT_DIR / "final_table.parquet")
final_df.head()

## MVP отчёт: NER + Graph

- Каноникализация сущностей: единый `entity_id`, `entity_display`, алиасы + лемматизация LOC и эвристика PER (однословный как фамилия, склейка с полными ФИО).
- Метрики (см. вывод `ner_quality` выше): `raw_total` → `kept_total`, `raw_unique` → `canon_unique`, `noise_ratio`, топ сущностей по числу алиасов.
- Примеры склеек (при наличии в данных): "Киев/Киева/Киеве" → "Киев", "Путин/Владимир Путин" → "Владимир Путин".
- Граф: добавлены узлы ENTITY/CHANNEL/CLUSTER(TOPIC) (+MSG опционально), связи публикации, тематические упоминания, co-mentions внутри темы/окна, фильтры по весу и top-N, IDF для частых сущностей.
- Артефакты: `entity_aliases.parquet`, обновлённый `doc_entities.parquet`, `nodes.parquet`, `edges.parquet`, `graph_metrics.parquet`, `theme_entity_graph.html`.

## J) Чек-лист качества

- Категории: список и распределение, примеры ошибок
- Тональность: sanity-check и распределение
- Кластеры: noise_ratio, largest_cluster_ratio, примеры тем, признаки мега-кластера
- Сущности: top PER/ORG/LOC, качество нормализации
- Граф: есть ли внятные связи и “центральные” сущности

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------
# 0) Safe defaults (если в ноутбуке нет констант)
# ---------------------------
EMBEDDING_MODEL = globals().get("EMBEDDING_MODEL", "ai-forever/sbert_large_nlu_ru")
WINDOW_HOURS = globals().get("WINDOW_HOURS", 6)
CONFIGS = globals().get("CONFIGS", None)  # если в твоей версии CONFIGS нет — просто не будем передавать

# ---------------------------
# 1) Helpers for NER
# ---------------------------
def _is_segmenter(obj) -> bool:
    return obj is not None and ("segment" in type(obj).__name__.lower())

def _is_ner(obj) -> bool:
    name = type(obj).__name__.lower()
    return ("ner" in name) or ("tagger" in name)

def _resolve_segmenter_and_ner(init_ner_result):
    """
    Возвращает (segmenter, ner) для сигнатуры extract_entities(text, segmenter, ner).
    Поддерживает init_ner(), который возвращает tuple/list с разными объектами,
    или один объект (ner).
    """
    segmenter = None
    ner = None

    if isinstance(init_ner_result, (tuple, list)):
        items = list(init_ner_result)

        for x in items:
            if _is_segmenter(x):
                segmenter = x
                break

        for x in items:
            if _is_ner(x) and x is not segmenter:
                ner = x
                break

        if segmenter is None and len(items) == 2:
            a, b = items
            if _is_ner(a) and not _is_ner(b):
                ner, segmenter = a, b
            elif _is_ner(b) and not _is_ner(a):
                ner, segmenter = b, a
            else:
                segmenter, ner = a, b
    else:
        ner = init_ner_result

    if segmenter is None:
        try:
            from natasha import Segmenter
            segmenter = Segmenter()
        except Exception as e:
            raise ValueError(
                "Не удалось получить segmenter из init_ner() и не удалось создать natasha.Segmenter(). "
                f"Ошибка: {e}"
            )

    if ner is None:
        raise ValueError(
            "Не удалось получить NER-модель (ner/tagger) из init_ner(). "
            "Проверь, что init_ner() возвращает объект NewsNERTagger/NERTagger."
        )

    return segmenter, ner

def _parse_extract_entities_output(res):
    """Приводит выход extract_entities(...) к формату списков PER/ORG/LOC."""
    per, org, loc = [], [], []

    def _add(t, v):
        if not v:
            return
        if isinstance(v, (list, tuple, set)):
            vals = list(v)
        else:
            vals = [v]
        vals = [str(x).strip() for x in vals if str(x).strip()]
        if t == "PER":
            per.extend(vals)
        elif t == "ORG":
            org.extend(vals)
        elif t == "LOC":
            loc.extend(vals)

    if res is None:
        return {"PER": [], "ORG": [], "LOC": []}

    if isinstance(res, dict):
        # вариант {'PER': [...], 'ORG': [...], 'LOC': [...]}
        if any(k in res for k in ("PER", "ORG", "LOC")):
            _add("PER", res.get("PER", []))
            _add("ORG", res.get("ORG", []))
            _add("LOC", res.get("LOC", []))
        # вариант {'entities': [{'type': 'PER', 'text': '...'}, ...]}
        elif "entities" in res and isinstance(res["entities"], (list, tuple)):
            for e in res["entities"]:
                if not isinstance(e, dict):
                    continue
                t = (e.get("type") or e.get("tag") or e.get("label") or "").upper()
                txt = e.get("text") or e.get("name") or e.get("value")
                if t in ("PER", "PERSON"):
                    _add("PER", txt)
                elif t in ("ORG", "ORGANIZATION"):
                    _add("ORG", txt)
                elif t in ("LOC", "GPE", "LOCATION"):
                    _add("LOC", txt)
        return {"PER": sorted(set(per)), "ORG": sorted(set(org)), "LOC": sorted(set(loc))}

    if isinstance(res, (list, tuple)):
        for e in res:
            if isinstance(e, dict):
                t = (e.get("type") or e.get("tag") or e.get("label") or "").upper()
                txt = e.get("text") or e.get("name") or e.get("value")
            else:
                t = str(getattr(e, "type", getattr(e, "tag", ""))).upper()
                txt = getattr(e, "text", getattr(e, "value", str(e)))
            if t in ("PER", "PERSON"):
                _add("PER", txt)
            elif t in ("ORG", "ORGANIZATION"):
                _add("ORG", txt)
            elif t in ("LOC", "GPE", "LOCATION"):
                _add("LOC", txt)
        return {"PER": sorted(set(per)), "ORG": sorted(set(org)), "LOC": sorted(set(loc))}

    return {"PER": [], "ORG": [], "LOC": []}


# ---------------------------
# 2) Вход: один пост
# ---------------------------
text = (
    "мой прогноз на 2036 союзное государство россия и китай станет единственной "
    "сверх державой на планете остальные по тем или иным причинам уйдут с верхушки"
)

raw_df = pd.DataFrame([{
    "post_id": 1,
    "channel": "manual_test",
    "date": pd.to_datetime("2026-02-16 12:00:00"),
    "text": text,
    "title": None,
    "topic_label": np.nan,       # принудительно zero-shot
    "sentiment_label": np.nan,   # принудительно pretrained
}])

# ---------------------------
# 3) Очистка
# ---------------------------
df = clean_data(raw_df)
display(df[["post_id", "text_clean"]])

# ---------------------------
# 4) Topic
# ---------------------------
topic_pred_df, topic_model_used = train_or_zero_shot_topic(
    df, text_col="text_clean", label_col="topic_label"
)
df = df.merge(topic_pred_df, on="post_id", how="left", suffixes=("", "_pred"))
if "topic_label_pred" in df.columns:
    df["topic_label"] = df["topic_label_pred"]
    df.drop(columns=["topic_label_pred"], inplace=True)

# ---------------------------
# 5) Sentiment
# ---------------------------
sent_pred_df, sent_model_used = train_or_pretrained_sentiment(
    df, text_col="text_clean", label_col="sentiment_label"
)
df = df.merge(sent_pred_df, on="post_id", how="left", suffixes=("", "_pred"))
if "sentiment_label_pred" in df.columns:
    df["sentiment_label"] = df["sentiment_label_pred"]
    df.drop(columns=["sentiment_label_pred"], inplace=True)

# ---------------------------
# 6) NER (extract_entities(text, segmenter, ner))
# ---------------------------
init_res = init_ner()
segmenter, ner = _resolve_segmenter_and_ner(init_res)

rows = []
for _, r in df.iterrows():
    res = extract_entities(text=r["text_clean"], segmenter=segmenter, ner=ner)
    parsed = _parse_extract_entities_output(res)
    rows.append({
        "post_id": r["post_id"],
        "PER": parsed["PER"],
        "ORG": parsed["ORG"],
        "LOC": parsed["LOC"],
    })

ent_df = pd.DataFrame(rows)
df = df.merge(ent_df, on="post_id", how="left")

# ---------------------------
# 7) Embeddings (устойчивый вызов при разных сигнатурах)
# ---------------------------
try:
    embeddings = compute_embeddings(df, text_col="text_clean", model_name=EMBEDDING_MODEL)
except TypeError:
    # если model_name или text_col не поддерживаются
    try:
        embeddings = compute_embeddings(df, text_col="text_clean")
    except TypeError:
        # если функция ожидает список текстов
        embeddings = compute_embeddings(df["text_clean"].tolist())

# ---------------------------
# 8) Clustering (для 1 текста — ставим -1, иначе вызываем cluster_themes)
# ---------------------------
if len(df) < 5 or "cluster_themes" not in globals():
    bucketed_df = df.copy()
    bucketed_df["cluster_id"] = -1
    bucketed_df["cluster_prob"] = 1.0
    best_cfg = None
else:
    # пробуем разные варианты сигнатуры
    try:
        if CONFIGS is None:
            bucketed_df, labels, probs, bucket_models, metrics_df, best_cfg, best_cfg_row = cluster_themes(
                df=df, embeddings=embeddings, window_hours=WINDOW_HOURS
            )
        else:
            bucketed_df, labels, probs, bucket_models, metrics_df, best_cfg, best_cfg_row = cluster_themes(
                df=df, embeddings=embeddings, window_hours=WINDOW_HOURS, configs=CONFIGS
            )
    except TypeError:
        # если configs/window_hours называются иначе, просто вызовем минимально
        bucketed_df, labels, probs, bucket_models, metrics_df, best_cfg, best_cfg_row = cluster_themes(
            df=df, embeddings=embeddings
        )

    bucketed_df["cluster_id"] = labels
    bucketed_df["cluster_prob"] = probs

# ---------------------------
# 9) Финальная витрина
# ---------------------------
# format_entities может ожидать PER/ORG/LOC — оставляем
bucketed_df["top_entities"] = bucketed_df.apply(format_entities, axis=1)

final_cols = [
    "post_id", "channel", "date",
    "text_clean",
    "topic_label", "topic_score",
    "sentiment_label", "sentiment_score",
    "cluster_id", "cluster_prob",
    "PER", "ORG", "LOC",
    "top_entities",
]
display(bucketed_df[final_cols])

print("Topic model:", topic_model_used)
print("Sentiment model:", sent_model_used)
print("Embedding model:", EMBEDDING_MODEL)
print("Best clustering cfg:", best_cfg)
